In [14]:
def eval_growth_metrics_for_val_sims_from_zarr(
    val_sims,
    r_modes,
    lamdag_zarr_path="lamdag_pred_r9.zarr",
    tol_rel=1e-3,
    eps_max=1e-12,
    ip_zarr_path="ip_nodal.zarr",
    gx_key="snaps_SDV1_bottom",
    gy_key="snaps_SDV4_bottom",
):
    """
    Same as before, but lamdag_hist_pred is loaded from a Zarr store created by
    precompute_lamdag_for_val_sims instead of from the rollout dict.
    """

    # True nodal growth
    g_ip = zarr.open(ip_zarr_path, mode="r")
    lamgx_all = np.asarray(g_ip[gx_key][:], dtype=np.float64)  # (N_bottom, S)
    lamgy_all = np.asarray(g_ip[gy_key][:], dtype=np.float64)

    # Displacement snapshots (for mask)
    g_disp = zarr.open_group("displacements.zarr", mode="r")
    disp_snaps = np.asarray(g_disp["snapshots"], dtype=np.float64)

    # Mapping arrays
    (_, _, _, _,
     sim_index, time_vals,
     _, _,
     _, _) = load_raw_data(r_modes)

    # Geometry
    N_bottom = bottom_xy.shape[0]
    N_elem   = bottom_face_conn_local.shape[0]

    # Open lamdag store
    g_lam = zarr.open(lamdag_zarr_path, mode="r")

    all_growth_metrics = []

    for sim_id in val_sims:
        print(f"\n=== Growth metrics (disp-based mask) for sim {sim_id} ===")

        grp_name = f"sim_{sim_id}"
        if grp_name not in g_lam:
            raise KeyError(f"{grp_name} not found in {lamdag_zarr_path}. "
                           "Did you run precompute_lamdag_for_val_sims first?")

        g_sim = g_lam[grp_name]
        lamdag_hist_pred = np.asarray(g_sim["lamdag_hist_pred"][:], dtype=np.float64)  # (T_roll, N_elem, 2)
        T_roll = lamdag_hist_pred.shape[0]

        # Match sim → FE snapshots
        idx = np.where(sim_index == sim_id)[0]
        t_s = time_vals[idx]
        order = np.argsort(t_s)
        idx_sorted = idx[order]

        T_eff = min(T_roll, len(idx_sorted))
        if T_eff < T_roll or T_eff < len(idx_sorted):
            print(f"  [WARN] Truncating trajectories from {T_roll} to {T_eff} steps.")
        idx_sorted_eff = idx_sorted[:T_eff]
        lamdag_eff = lamdag_hist_pred[:T_eff, :, :]  # (T_eff, N_elem, 2)

        # ---- build displacement-based mask from final uz ----
        snap_idx_final = idx_sorted_eff[T_eff - 1]
        u_true_flat = disp_snaps[:, snap_idx_final]
        ux_t, uy_t, uz_t = split_disp_block(u_true_flat)
        assert uz_t.shape[0] == N_bottom

        uz_abs = np.abs(uz_t)
        max_uz = np.max(uz_abs)
        if max_uz < eps_max:
            print("  [WARN] Final uz ~0; using full field for mask.")
            mask = np.ones_like(uz_t, dtype=bool)
        else:
            thresh = tol_rel * max_uz
            mask = uz_abs >= thresh
        if not np.any(mask):
            mask = np.ones_like(uz_t, dtype=bool)
        n_mask = int(np.count_nonzero(mask))
        print(f"  Mask keeps {n_mask}/{uz_t.size} bottom nodes.")

        # ---- true nodal growth for this sim ----
        lamgx_true = lamgx_all[:, idx_sorted_eff].T  # (T_eff, N_bottom)
        lamgy_true = lamgy_all[:, idx_sorted_eff].T

        # ---- map element lamdag → nodal growth (same as before) ----
        lamgx_pred_nodal = np.zeros((T_eff, N_bottom), dtype=np.float64)
        lamgy_pred_nodal = np.zeros((T_eff, N_bottom), dtype=np.float64)
        counts = np.zeros(N_bottom, dtype=np.int32)

        for k in range(T_eff):
            lamgx_pred_nodal[k, :] = 0.0
            lamgy_pred_nodal[k, :] = 0.0
            counts[:] = 0

            lam_k = lamdag_eff[k]  # (N_elem, 2)
            for e in range(N_elem):
                lamgx_e = lam_k[e, 0]
                lamgy_e = lam_k[e, 1]
                nodes_e = bottom_face_conn_local[e]
                for nid in nodes_e:
                    lamgx_pred_nodal[k, nid] += lamgx_e
                    lamgy_pred_nodal[k, nid] += lamgy_e
                    counts[nid] += 1

            nonzero = counts > 0
            lamgx_pred_nodal[k, nonzero] /= counts[nonzero]
            lamgy_pred_nodal[k, nonzero] /= counts[nonzero]

            zero = ~nonzero
            if np.any(zero):
                lamgx_pred_nodal[k, zero] = 1.0
                lamgy_pred_nodal[k, zero] = 1.0

        # ---- λ_g - 1, apply same mask, compute metrics ----
        gx_true_dev = lamgx_true - 1.0
        gy_true_dev = lamgy_true - 1.0
        gx_pred_dev = lamgx_pred_nodal - 1.0
        gy_pred_dev = lamgy_pred_nodal - 1.0

        gx_true_dev_m = gx_true_dev[:, mask]
        gy_true_dev_m = gy_true_dev[:, mask]
        gx_pred_dev_m = gx_pred_dev[:, mask]
        gy_pred_dev_m = gy_pred_dev[:, mask]

        g_true_dev_all = np.concatenate([gx_true_dev_m, gy_true_dev_m], axis=1)
        g_pred_dev_all = np.concatenate([gx_pred_dev_m, gy_pred_dev_m], axis=1)

        g_true_flat = g_true_dev_all.reshape(-1)
        g_pred_flat = g_pred_dev_all.reshape(-1)
        g_err_flat  = g_pred_flat - g_true_flat

        g_L2_all       = float(np.linalg.norm(g_err_flat))
        g_rel_all      = float(g_L2_all / (np.linalg.norm(g_true_flat) + 1e-12))
        g_mean_abs_all = float(np.mean(np.abs(g_err_flat)))
        g_rel_mean_all = float(
            np.mean(np.abs(g_err_flat) / (np.abs(g_true_flat) + 1e-12))
        )

        g_true_final = g_true_dev_all[-1, :]
        g_pred_final = g_pred_dev_all[-1, :]
        g_err_final  = g_pred_final - g_true_final

        g_L2_final  = float(np.linalg.norm(g_err_final))
        g_rel_final = float(
            g_L2_final / (np.linalg.norm(g_true_final) + 1e-12)
        )

        print(f"  Growth L2 (all times, masked)   = {g_L2_all:.4e}")
        print(f"  Growth rel (all times, masked)  = {g_rel_all:.4e}")
        print(f"  Growth mean |err| (all, masked) = {g_mean_abs_all:.4e}")
        print(f"  Growth mean rel (all, masked)   = {g_rel_mean_all:.4e}")
        print(f"  Growth L2 (final, masked)       = {g_L2_final:.4e}")
        print(f"  Growth rel (final, masked)      = {g_rel_final:.4e}")

        all_growth_metrics.append({
            "sim_id": sim_id,
            "n_mask": n_mask,
            "g_L2_all": g_L2_all,
            "g_rel_all": g_rel_all,
            "g_mean_abs_all": g_mean_abs_all,
            "g_rel_mean_all": g_rel_mean_all,
            "g_L2_final": g_L2_final,
            "g_rel_final": g_rel_final,
        })

    return all_growth_metrics


In [15]:
import zarr
import numpy as np

In [20]:
growth_metrics_r9 = eval_growth_metrics_for_val_sims_from_zarr(
    val_sims,
    r_modes=9,
    lamdag_zarr_path="lamdag_pred_r9.zarr",
)

NameError: name 'U_LATENT_TEMPLATE' is not defined

In [5]:
val_sims = [210, 188, 249, 7, 584, 921, 294, 924, 85, 316, 291, 760, 889, 279, 303, 396, 580, 200, 267, 794, 682, 310, 732, 643, 369, 832, 280, 147, 172, 196, 894, 227, 566, 733, 343, 585, 344, 145, 352, 577, 458, 728, 916, 749, 161, 692, 660, 726, 880, 138, 5, 269, 373, 346, 876, 41, 657, 184, 134, 683, 353, 735, 536, 306, 606, 712, 761, 724, 447, 372, 48, 748, 859, 548, 388, 273, 900, 35, 177, 298, 785, 171, 590, 4, 314, 879, 508, 844, 699, 430, 50, 170, 389, 919, 663, 209, 744, 164, 561, 775, 57, 131, 551, 167, 543, 203, 520, 578, 55, 746, 521, 461, 453, 840, 847, 103, 804, 597, 814, 328, 793, 52, 852, 849, 338, 54, 117, 831, 769, 374, 204, 31, 421, 553, 36, 702, 798, 348, 252, 313, 100, 911, 621, 274, 270, 624, 600, 791, 78, 743, 345, 79, 162, 336, 235, 568, 616, 329, 351, 91, 807, 90, 75, 656, 898, 531, 825, 392, 260, 829, 221, 813, 776, 511, 655, 802, 545, 677, 861, 261, 13, 74, 185, 229, 437]

In [19]:
def load_raw_data(r_modes: int):
    """
    Load raw data for a given r_modes (per field).
    Returns:
        U_lat        : (S, r)
        Lx_lat       : (S, r)
        Ly_lat       : (S, r)
        volume_snap  : (S,)   actual expander volume per snapshot
        sim_index    : (S,)
        time_vals    : (S,)
        volume_SP    : (S,)   volume setpoint per snapshot
        design_all   : (S, 7)
        dt_from_data : float  mean dt estimated from time_vals (for sanity)
        n_sims       : int
    """
    # --- open zarrs ---
    g_disp = zarr.open_group("displacements.zarr", mode="r")
    g_sdv  = zarr.open_group("ip_nodal.zarr",  mode="r")

    # paths for this r_modes
    u_path    = U_LATENT_TEMPLATE.format(r=r_modes)
    lamx_path = LAMGX_LATENT_TEMPLATE.format(r=r_modes)
    lamy_path = LAMGY_LATENT_TEMPLATE.format(r=r_modes)

    # latent coeffs (R, S) -> (S, R)
    U_lat  = np.asarray(g_disp[u_path],    dtype=np.float32).T  # (S, r)
    Lx_lat = np.asarray(g_sdv[lamx_path],  dtype=np.float32).T
    Ly_lat = np.asarray(g_sdv[lamy_path],  dtype=np.float32).T

    # mapping arrays
    mgrp = g_disp["mapping"]
    sim_index   = np.asarray(mgrp["sim_index"],   dtype=np.int64)    # (S,)
    time_vals   = np.asarray(mgrp["time_vals"],   dtype=np.float64)  # (S,)
    volume_SP   = np.asarray(mgrp["volume_setpoint_per_snapshot"], dtype=np.float32)  # (S,)
    frame_index = np.asarray(mgrp["frame_index"], dtype=np.int64)    # (S,)

    # design params
    g_design   = zarr.open_group(DESIGN_ZARR_PATH, mode="r")
    design_all = np.asarray(g_design[DESIGN_DATASET], dtype=np.float32)  # (S, 7)

    S = U_lat.shape[0]
    assert Lx_lat.shape[0] == S and Ly_lat.shape[0] == S
    assert sim_index.shape[0] == S == time_vals.shape[0]
    assert volume_SP.shape[0] == S == design_all.shape[0]

    # --- load expander volume snapshots from expander_volume/cvol ---
    cvol_arr = np.asarray(g_disp["expander_volume/cvol"], dtype=np.float32)

    if cvol_arr.ndim == 1:
        # already per snapshot
        assert cvol_arr.shape[0] == S, "cvol length != #snapshots"
        volume_snap = cvol_arr
    elif cvol_arr.ndim == 2:
        # cvol[sim, frame]
        n_sims_cvol, n_frames_cvol = cvol_arr.shape
        if sim_index.max() >= n_sims_cvol:
            raise ValueError(f"sim_index max {sim_index.max()} >= cvol n_sims {n_sims_cvol}")
        if frame_index.max() >= n_frames_cvol:
            raise ValueError(f"frame_index max {frame_index.max()} >= cvol n_frames {n_frames_cvol}")
        volume_snap = cvol_arr[sim_index, frame_index]
    else:
        raise ValueError(f"Unexpected cvol ndim={cvol_arr.ndim}")

    # estimate dt from data (sanity only)
    unique_sims = np.unique(sim_index)
    dts = []
    for s in unique_sims:
        idx_s = np.where(sim_index == s)[0]
        if idx_s.size > 1:
            t_s = np.sort(time_vals[idx_s])
            dts.append(np.mean(np.diff(t_s)))
    dt_from_data = float(np.mean(dts)) if dts else FIXED_DT

    print(f"[r={r_modes}] Raw data loaded.")
    print(f"  U_lat:       {U_lat.shape}")
    print(f"  Lx_lat:      {Lx_lat.shape}")
    print(f"  Ly_lat:      {Ly_lat.shape}")
    print(f"  volume_snap: {volume_snap.shape}")
    print(f"  dt_from_data ≈ {dt_from_data:.4f} days, FIXED_DT = {FIXED_DT:.4f} days")

    return (U_lat, Lx_lat, Ly_lat,
            volume_snap, sim_index, time_vals,
            volume_SP, design_all,
            dt_from_data, len(unique_sims))
